In [23]:
include("../src/EBC.jl")
include("../../BeamToyModel/src/BeamToyModel.jl")
using Base.Threads
using Statistics
using NPZ

In [24]:
nside_in = 32
res = Resolution(nside_in)
lmax_in = 3nside_in-1
fwhm = deg2rad(1.0)
blm_harmonic = BeamToyModel.gaussian_harmonic(lmax_in, fwhm, mmax = 2)
;

In [340]:
cs = ConvolutionSky(lmax = lmax_in, numofsky = 1)
@show fieldnames(typeof(cs))
cb = ConvolutionBeam(lmax = lmax_in, mmax =2, numofbeams = 2)
@show fieldnames(typeof(cb))
cc = ConvolutionCalculate(nside_output = nside_in, lstart = 0, mmax_calculate = 2)
@show fieldnames(typeof(cc))

fieldnames(typeof(cs)) = (:numofsky, :lmax, :alm)
fieldnames(typeof(cb)) = (:numofbeams, :lmax, :mmax, :blm)
fieldnames(typeof(cc)) = (:nside_output, :lstart, :lstop, :mmax_calculate, :HWP)


(:nside_output, :lstart, :lstop, :mmax_calculate, :HWP)

In [330]:
cc.HWP

false

In [342]:
#cb.mmax = 2
cb.blm[:,:,1] = hcat(blm_harmonic.blmT.alm,blm_harmonic.blmE.alm,blm_harmonic.blmB.alm)
cb.blm[:,:,2] = hcat(blm_harmonic.blmT.alm,blm_harmonic.blmE.alm,blm_harmonic.blmB.alm)

;

In [343]:
alm_in = npzread("./alm=CMB=r0=nside32=seed1.npy")
cs.alm[1,:,:] = alm_in

3×4656 Matrix{ComplexF64}:
 0.0+0.0im  0.0+0.0im     -17.2712+0.0im  …     0.24645+1.48487im
 0.0+0.0im  0.0+0.0im    -0.216198+0.0im      0.0058999-0.00792962im
 0.0+0.0im  0.0+0.0im  -0.00188638+0.0im     4.92514e-5-0.00116108im

In [344]:
pix_idx = 60
θ_pix,φ_pix = Healpix.pix2angRing(res, pix_idx)

(0.12766426866687355, 6.126105674500097)

In [345]:
nptg = 10
θ = θ_pix
φ = φ_pix
dθ = rand(nptg)*1e-4*0
dφ = rand(nptg)*1e-4*0
ψ = rand(nptg)*2pi

10-element Vector{Float64}:
 4.4114169468536035
 4.86637760463165
 2.5087668110616375
 0.7291654599604898
 4.221819546922172
 0.7531021475294992
 4.994275529341328
 1.022075394151714
 3.257610315379102
 3.0242909384929955

In [369]:
function calc_local_euiler_angles(res, pix_idx, φ, θ, ψ; tol=1e-12)
    alphas = zeros(size(θ))
    betas = zeros(size(θ))
    gammas = zeros(size(θ))
    θ_pix,φ_pix = Healpix.pix2angRing(res, pix_idx)
    dθ = θ .- θ_pix
    dφ = φ .- φ_pix
    for i in eachindex(θ)
        err, (alphas[i], betas[i], gammas[i]) = check_split(φ_pix, θ_pix, dφ[i], dθ[i], ψ[i])
        if err > tol
            error("check_split error at i=$i: err=$err > tol=$tol")
        end
    end
    return alphas, betas, gammas
end

calc_local_euiler_angles (generic function with 1 method)

In [347]:
function local_effective_wignerD_conj_reduced_formapmake(cb, cc, α, β, γ, ψ; τ::Int=5)
    n_beam = sum(2*min(l, cb.mmax) + 1 for l in cc.lstart:cc.lstop)
    n_sky = sum(2*l + 1 for l in cc.lstart:cc.lstop)
    D_temp = 0+0im
    D_beam = spzeros(ComplexF64, n_sky, n_beam)
    D_beam2 = spzeros(ComplexF64, n_sky, n_beam)
    D_beam3 = spzeros(ComplexF64, n_sky, n_beam)
    @inbounds for l in cc.lstart:cc.lstop
        n_ = min(l, cb.mmax)
        @inbounds for i in eachindex(α)
            phase = exp(-2im*ψ[i])
            phase2 = exp(-4im*ψ[i])
            @inbounds for m in -l:l
                m_idx = lmr_idx(l=l, m=m, lstart=cc.lstart, mmax=cc.lstop)
                # ★帯制限：|m-n| <= τ だけ回す
                n_lo = max(-n_, m - τ)
                n_hi = min( n_, m + τ)
                #@show n_lo, n_hi, n_
                @inbounds for n in n_lo:n_hi
                    sgn = pm(m, n)
                    n_idx = lmr_idx(l=l, m=n, lstart=cc.lstart, mmax=cb.mmax)
                    D_temp = sgn * WignerD.wignerDjmn(l, -m, -n, α[i], β[i], γ[i])
                    D_beam[m_idx, n_idx] += D_temp
                    D_beam2[m_idx, n_idx] += D_temp * phase
                    D_beam3[m_idx, n_idx] += D_temp * (phase2)
                end
            end
        end
    end
    return D_beam./length(α), D_beam2./length(α), D_beam3./length(α)
end


local_effective_wignerD_conj_reduced_formapmake (generic function with 1 method)

In [348]:
α_local, β_local, γ_local = calc_local_euiler_angles(res, pix_idx, φ.+dφ, θ.+dθ, ψ)
;

In [349]:
@time A = local_effective_wignerD(cb, cc, α_local, β_local, γ_local)
@time A_conj = local_effective_wignerD_conj(cb, cc, α_local, β_local, γ_local)
@time A2_conj = local_effective_wignerD_conj_reduced(cb, cc, α_local, β_local, γ_local, τ=10)
@time A2_conj2 = local_effective_wignerD_conj_reduced_formapmake(cb, cc, α_local, β_local, γ_local, ψ, τ=20)
#@time A2_conj22 = local_effective_wignerD_conj_reduced_moments(cb, cc, α_local, β_local, γ_local, ψ, τ=10)

;

  0.404320 seconds (25 allocations: 1.491 MiB)
  0.396306 seconds (25 allocations: 1.491 MiB)
  0.073731 seconds (26 allocations: 489.828 KiB)
  0.332711 seconds (123.99 k allocations: 11.679 MiB, 59.57% compilation time)


In [350]:
lD = local_effective_wignerD_conj_reduced_formapmake(cb, cc, α_local, β_local, γ_local, ψ, τ=5)
gD = global_wignerD_conj(cc, φ, θ, 0)

9216×9216 SparseMatrixCSC{ComplexF64, Int64} with 1179616 stored entries:
⎡⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠈⠻⢆⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠿⣧⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠿⣧⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠛⣤⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠿⣧⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠿⣧⣀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⠻⣦⡄⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⢻⣶⣀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⠻⣦⡄⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⢻⣶⎦

In [351]:
function convolver_1pixel(cs, cb, cc, alm_slice, blm_slice, globalD, localD)
    if cc.HWP == false
        result = zeros(ComplexF64, cs.numofsky, cb.numofbeams, 3)
        for i in 1:cs.numofsky
            for j in 1:cb.numofbeams
                result[i,j,1] = transpose(alm_slice[1,1,:])*globalD*localD[1]*blm_slice[:,1,1] + 1/2*transpose(alm_slice[1,2,:])*globalD*localD[1]*blm_slice[:,2,1] + 1/2* transpose(alm_slice[1,3,:])*globalD*localD[1]*blm_slice[:,3,1]
                result[i,j,2] = conj(transpose(alm_slice[1,1,:])*globalD*localD[2]*blm_slice[:,1,1]) + 1/2*transpose(alm_slice[1,2,:])*globalD*localD[1]*blm_slice[:,2,1] + 1/2*conj(transpose(alm_slice[1,3,:])*globalD*localD[3]*blm_slice[:,3,1])
                result[i,j,3] = conj(result[i,j,2])
            end
        end
    end
    return result
end

convolver_1pixel (generic function with 2 methods)

In [354]:
alm_slice = slice_spin_alm_by_l(cs, cc)
blm_slice = slice_spin_blm_by_l(cb, cc);

In [365]:
@time test = convolver_1pixel(cs, cb, cc, alm_slice, blm_slice, gD, lD)

  0.057194 seconds (91 allocations: 5.197 MiB)


1×2×3 Array{ComplexF64, 3}:
[:, :, 1] =
 -31.1883+9.67778e-13im  -31.1883+9.67778e-13im

[:, :, 2] =
 3.78183-7.75134im  3.78183-7.75134im

[:, :, 3] =
 3.78183+7.75134im  3.78183+7.75134im

In [367]:
sizeof(alm_slice)         # 配列本体データのバイト数（要素部分）
Base.summarysize(alm_slice)*1e-6

0.442416

In [ ]:
using Base: format_bytes
println(format_bytes(sizeof(alm_slice)))
println(format_bytes(Base.summarysize(alm_slice)))


432.000 KiB
432.047 KiB


In [329]:
@show transpose(alm_[1,1,:])*gD*lD[1]*blm_[:,1,1] + 1/2*transpose(alm_[1,2,:])*gD*lD[1]*blm_[:,2,1] + 1/2* transpose(alm_[1,3,:])*gD*lD[1]*blm_[:,3,1]
@show conj(transpose(alm_[1,1,:])*gD*lD[2]*blm_[:,1,1]) + 1/2*transpose(alm_[1,2,:])*gD*lD[1]*blm_[:,2,1] + 1/2*conj(transpose(alm_[1,3,:])*gD*lD[3]*blm_[:,3,1])
@show transpose(alm_[1,1,:])*gD*lD[2]*blm_[:,1,1] + 1/2*transpose(alm_[1,2,:])*gD*lD[3]*blm_[:,2,1] + 1/2*transpose(alm_[1,3,:])*gD*lD[1]*blm_[:,3,1]


transpose(alm_[1, 1, :]) * gD * lD[1] * blm_[:, 1, 1] + (1 / 2) * transpose(alm_[1, 2, :]) * gD * lD[1] * blm_[:, 2, 1] + (1 / 2) * transpose(alm_[1, 3, :]) * gD * lD[1] * blm_[:, 3, 1] = -31.17832202906844 + 9.65502634439619e-13im
conj(transpose(alm_[1, 1, :]) * gD * lD[2] * blm_[:, 1, 1]) + (1 / 2) * transpose(alm_[1, 2, :]) * gD * lD[1] * blm_[:, 2, 1] + (1 / 2) * conj(transpose(alm_[1, 3, :]) * gD * lD[3] * blm_[:, 3, 1]) = 14.296221783598297 - 2.1860739126419917im
transpose(alm_[1, 1, :]) * gD * lD[2] * blm_[:, 1, 1] + (1 / 2) * transpose(alm_[1, 2, :]) * gD * lD[3] * blm_[:, 2, 1] + (1 / 2) * transpose(alm_[1, 3, :]) * gD * lD[1] * blm_[:, 3, 1] = 14.290603029960522 + 2.191205778888753im


14.290603029960522 + 2.191205778888753im

In [360]:
@show test[:,1,:]
@show test[:,2,:]

test[:, 1, :] = ComplexF64[-31.18833634691746 + 9.67778374799666e-13im 3.7818349755211136 - 7.751339464295756im 3.7818349755211136 + 7.751339464295756im]
test[:, 2, :] = ComplexF64[-31.18833634691746 + 9.67778374799666e-13im 3.7818349755211136 - 7.751339464295756im 3.7818349755211136 + 7.751339464295756im]


1×3 Matrix{ComplexF64}:
 -31.1883+9.67778e-13im  3.78183-7.75134im  3.78183+7.75134im

In [361]:
@show conj(transpose(alm_[1,1,:])*gD*lD[2]*blm_[:,1,1]) + 1/2*transpose(alm_[1,2,:])*gD*lD[1]*blm_[:,2,1] + 1/2*conj(transpose(alm_[1,3,:])*gD*lD[3]*blm_[:,3,1])
@show transpose(alm_[1,1,:])*gD*lD[2]*blm_[:,1,1] + 1/2*transpose(alm_[1,2,:])*gD*lD[3]*blm_[:,2,1] + 1/2*transpose(alm_[1,3,:])*gD*lD[1]*blm_[:,3,1]

conj(transpose(alm_[1, 1, :]) * gD * lD[2] * blm_[:, 1, 1]) + (1 / 2) * transpose(alm_[1, 2, :]) * gD * lD[1] * blm_[:, 2, 1] + (1 / 2) * conj(transpose(alm_[1, 3, :]) * gD * lD[3] * blm_[:, 3, 1]) = 3.7818349755211136 - 7.751339464295756im
transpose(alm_[1, 1, :]) * gD * lD[2] * blm_[:, 1, 1] + (1 / 2) * transpose(alm_[1, 2, :]) * gD * lD[3] * blm_[:, 2, 1] + (1 / 2) * transpose(alm_[1, 3, :]) * gD * lD[1] * blm_[:, 3, 1] = 3.7773197931021154 + 7.744746376529789im


3.7773197931021154 + 7.744746376529789im

In [362]:
mean(i.*exp.(-2im*ψ[:]))

3.7783936720593987 + 7.751642777830966im

In [363]:
conj(transpose(alm_[1,1,:])*gD*lD[2]*blm_[:,1,1])

3.7783936720597033 - 7.7516427778309795im

In [364]:
i = maps.i[pix_idx]
p = maps.q[pix_idx]+im*maps.u[pix_idx]
pp = conj(p)
@show mean(i .+ 1/2 .* p .* exp.(-2im*ψ[:]) .+ 1/2 .* pp .*exp.(2im*ψ[:]))
@show mean(i.*exp.(2im*ψ[:]) .+ 1/2 .* p  .+ 1/2 .* pp .*exp.(4im*ψ[:]))
@show mean(i.*exp.(-2im*ψ[:]) .+ 1/2 .* p.*exp.(-4im*ψ[:])  .+ 1/2 .* pp )


mean((i .+ ((1 / 2) .* p) .* exp.((-2im) * ψ[:])) .+ ((1 / 2) .* pp) .* exp.((2im) * ψ[:])) = -31.188335727408315 + 0.0im
mean((i .* exp.((2im) * ψ[:]) .+ (1 / 2) .* p) .+ ((1 / 2) .* pp) .* exp.((4im) * ψ[:])) = 3.764877533210766 - 7.750931384199028im
mean((i .* exp.((-2im) * ψ[:]) .+ ((1 / 2) .* p) .* exp.((-4im) * ψ[:])) .+ (1 / 2) .* pp) = 3.764877533210766 + 7.750931384199028im


3.764877533210766 + 7.750931384199028im

In [312]:
I = maps.i[pix_idx]


LoadError: cannot assign a value to imported variable LinearAlgebra.I from module Main

In [305]:
transpose(alm_[1,2,:])*gD*lD[1]*blm_[:,2,1]

0.012833352066202938 + 0.0031386833456924874im

In [283]:
@show transpose(alm_[1,2,:])*gD*lD[1]*blm_[:,2,1]
@show mean((maps.q[pix_idx]+im*maps.u[pix_idx]).*exp.(-2im*ψ[:]))

transpose(alm_[1, 2, :]) * gD * lD[1] * blm_[:, 2, 1] = 0.012833352066202938 + 0.0031386833456924874im
mean((maps.q[pix_idx] + im * maps.u[pix_idx]) .* exp.((-2im) * ψ[:])) = 0.012836172312913657 + 0.0031393731001219797im


0.012836172312913657 + 0.0031393731001219797im

In [284]:
transpose(alm_[1,1,:])*gD*lD[1]*blm_[:,1,1]

-31.19115538113464 + 9.669458564511667e-13im

In [285]:
transpose(alm_[1,2,:])*gD*lD[1]*blm_[:,2,1]

0.012833352066202938 + 0.0031386833456924874im

In [277]:
mean((maps.i[pix_idx].*exp.(2im*ψ[:]) .+ 1/2*(maps.q[pix_idx]+im*maps.u[pix_idx]).*exp.(0im*ψ[:]) .+ 1/2*(maps.q[pix_idx]-im*maps.u[pix_idx]).*exp.(4im*ψ[:])))


14.267212776024525 - 2.1889014716322173im

In [212]:
@show transpose(alm_[1,1,:])*B*A2_conj*blm_[:,1,1]
@show transpose(alm_[1,2,:])*B*A2_conj*blm_[:,2,1]
@show transpose(alm_[1,3,:])*B*A_conj*blm_[:,3,1]

transpose(alm_[1, 1, :]) * B * A2_conj * blm_[:, 1, 1] = -31.19365185302878 + 1.4133932046467997e-12im
transpose(alm_[1, 2, :]) * B * A2_conj * blm_[:, 2, 1] = 0.00555872545167838 - 0.002084933078593232im
transpose(alm_[1, 3, :]) * B * A_conj * blm_[:, 3, 1] = 0.00555872545167647 + 0.0020849330785970726im


0.00555872545167647 + 0.0020849330785970726im

In [160]:
@show maps.i[pix_idx]
@show maps.q[pix_idx]
@show maps.u[pix_idx]

maps.i[pix_idx] = -31.1911553811341
maps.q[pix_idx] = -0.02841467270396214
maps.u[pix_idx] = -0.002504444878711355


-0.002504444878711355

In [161]:
mean((maps.q[pix_idx]+im*maps.u[pix_idx]).*exp.(-2im*ψ[:]))

0.005727513527892027 - 0.0018337264152531916im

In [165]:
A2_conj2

9216×474 SparseMatrixCSC{ComplexF64, Int64} with 9510 stored entries:
⎡⢧⠀⠀⎤
⎢⢸⠀⠀⎥
⎢⢸⠀⠀⎥
⎢⢸⡀⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⠳⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢘⠀⎥
⎢⠀⠸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢐⠀⎥
⎢⠀⠸⠀⎥
⎢⠀⠨⠀⎥
⎢⠀⠀⡅⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⠅⎥
⎣⠀⠀⠅⎦

In [127]:
alm_smooth = npzread("alm_smooth=CMB=r0=nside32=seed1.npy")

alm2 = Healpix.Alm(cs.lmax, cs.lmax, zeros(ComplexF64, Healpix.numberOfAlms(cs.lmax, cs.lmax)))
alm2.alm = alm_smooth[1,:]
mT = Healpix.alm2map(alm2, nside_in)   # 戻りは Healpix.Map

nalm  = Healpix.numberOfAlms(cs.lmax, cs.lmax)
almT = Healpix.Alm(cs.lmax, cs.lmax, zeros(ComplexF64, nalm))
almE = Healpix.Alm(cs.lmax, cs.lmax, zeros(ComplexF64, nalm))
almB = Healpix.Alm(cs.lmax, cs.lmax, zeros(ComplexF64, nalm))
almT.alm = alm_smooth[1,:]
almE.alm = alm_smooth[2,:]
almB.alm = alm_smooth[3,:]

maps= alm2map([almT, almE, almB], nside_in)

PolarizedHealpixMap{Float64, RingOrder, Vector{Float64}}([-58.49604137595402, -5.403537747114228, -11.298414133610322, -56.314667251724096, -59.699267217576754, -85.66622597795597, -71.17192250601221, -24.215049528928265, -74.16297882035632, -33.33706956041863  …  25.614428728708468, 142.1401271522759, 106.63577208567405, 222.1715373562783, 130.88070501569757, 156.216220113293, 62.39665256415908, 123.58634220400221, 167.54744250106972, 75.16966680268094], [0.3841684359262724, -0.2283283590288716, -0.588348244867616, -0.8973271051817646, -0.0369796019997603, 0.053658932027735334, -0.04304553931855282, -0.028340964839986604, -0.5793540400447466, 0.11914016822512552  …  0.034733190068783434, -0.4229562186824516, -0.01814141514429357, 0.036842700099326456, -0.20377726778227395, -0.06944079633454092, 0.4894117687322749, -0.43221459323236966, 0.013511342306182589, -0.2992332489751508], [-0.12281782625132895, 0.2541801993465586, 0.3691346888610124, -0.29191966435191946, 0.22773662495005553, 0

In [106]:
exp(-2im*ψ[1])

0.6234898018587336 - 0.7818314824680298im

In [105]:
v = vec(alm_[1][1,:])
@show size(v) size(B) size(A2_conj)


LoadError: MethodError: no method matching getindex(::ComplexF64, ::Int64, ::Colon)

[0mClosest candidates are:
[0m  getindex(::Number, ::Integer)
[0m[90m   @[39m [90mBase[39m [90m[4mnumber.jl:96[24m[39m
[0m  getindex(::Number, [91m::Integer...[39m)
[0m[90m   @[39m [90mBase[39m [90m[4mnumber.jl:101[24m[39m
[0m  getindex(::Number)
[0m[90m   @[39m [90mBase[39m [90m[4mnumber.jl:95[24m[39m
[0m  ...


In [47]:
alm_[1]

0.0 + 0.0im

In [43]:
alm_[:,1,1]*B

LoadError: DimensionMismatch: 

In [128]:
blm_

474×3×1 Array{ComplexF64, 3}:
[:, :, 1] =
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
 -1.0-1.0im   2.0+0.0im   0.0+2.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
  1.0+1.0im  -2.0-0.0im  -0.0-2.0im
 -1.0-1.0im   2.0+0.0im   0.0+2.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
  1.0+1.0im  -2.0-0.0im  -0.0-2.0im
 -1.0-1.0im   2.0+0.0im   0.0+2.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
     ⋮                   
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
  1.0+1.0im  -2.0-0.0im  -0.0-2.0im
 -1.0-1.0im   2.0+0.0im   0.0+2.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
  1.0+1.0im  -2.0-0.0im  -0.0-2.0im
 -1.0-1.0im   2.0+0.0im   0.0+2.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im

In [105]:
idx_in = alm_idx(l=cb.lmax, m=cb.lmax, lmax=cb.lmax, mmax = cb.mmax)

LoadError: ArgumentError: m must be in [0, mmax]

In [104]:
cb.mmax

2

In [ ]:
n = alm_idx(lmax = cb.lmax, mmax = cc.mmax_calculate, l=cb.lmax, m=cc.mmax_calculate)
blm_calc = Array{ComplexF64,3}(undef, n, 3, cb.numofbeams)

285×3×1 Array{ComplexF64, 3}:
[:, :, 1] =
  2.2749e-314+6.92672e-310im  …  6.92672e-310+6.92672e-310im
          NaN+6.92672e-310im     6.92669e-310+6.92672e-310im
          NaN+6.92672e-310im     6.92672e-310+6.92669e-310im
          NaN+6.92672e-310im     6.92672e-310+6.92672e-310im
          NaN+6.92672e-310im     6.92672e-310+6.92672e-310im
          NaN+6.92672e-310im  …  6.92672e-310+6.92672e-310im
          NaN+4.0e-323im         6.92672e-310+6.92672e-310im
 2.27484e-314+6.92672e-310im     6.92672e-310+6.92672e-310im
 2.27488e-314+6.92672e-310im     6.92672e-310+6.92672e-310im
          NaN+6.92672e-310im     6.92672e-310+6.92664e-310im
 2.27484e-314+6.92672e-310im  …  6.92672e-310+6.92672e-310im
 2.27484e-314+0.0im              6.92664e-310+6.92672e-310im
 2.27484e-314+6.92672e-310im     6.92672e-310+6.92672e-310im
             ⋮                ⋱  
     4.0e-323+0.0im              6.92672e-310+1.04e-322im
          NaN+1.50662e-312im     6.92672e-310+1.7e-322im
          0.0+6.

In [88]:
n = 10
blm_calc = Array{ComplexF64,3}(undef, 3, n, cb.numofbeams)


3×10×1 Array{ComplexF64, 3}:
[:, :, 1] =
 2.44636e-314+2.43256e-314im  …  2.45103e-314+2.45103e-314im
 2.43299e-314+2.46808e-314im     2.77884e-314+2.46808e-314im
 2.46808e-314+2.46808e-314im              0.0+2.78145e-314im

In [99]:
cb.blm

285×3×1 Array{ComplexF64, 3}:
[:, :, 1] =
 0.282095+0.0im       0.0+0.0im  0.0+0.0im
 0.488576+0.0im       0.0+0.0im  0.0+0.0im
 0.630679+0.0im       0.0+0.0im  0.0+0.0im
 0.746107+0.0im       0.0+0.0im  0.0+0.0im
  0.84582+0.0im       0.0+0.0im  0.0+0.0im
 0.934832+0.0im       0.0+0.0im  0.0+0.0im
  1.01593+0.0im       0.0+0.0im  0.0+0.0im
  1.09087+0.0im       0.0+0.0im  0.0+0.0im
  1.16081+0.0im       0.0+0.0im  0.0+0.0im
  1.22659+0.0im       0.0+0.0im  0.0+0.0im
  1.28882+0.0im       0.0+0.0im  0.0+0.0im
  1.34798+0.0im       0.0+0.0im  0.0+0.0im
  1.40444+0.0im       0.0+0.0im  0.0+0.0im
         ⋮                       
      0.0+0.0im  -1.50692+0.0im  0.0-1.50692im
      0.0+0.0im  -1.50875+0.0im  0.0-1.50875im
      0.0+0.0im  -1.51039+0.0im  0.0-1.51039im
      0.0+0.0im  -1.51186+0.0im  0.0-1.51186im
      0.0+0.0im  -1.51314+0.0im  0.0-1.51314im
      0.0+0.0im  -1.51424+0.0im  0.0-1.51424im
      0.0+0.0im  -1.51517+0.0im  0.0-1.51517im
      0.0+0.0im  -1.51592+0.0im  0.0

In [91]:
cb.blm

285×3×1 Array{ComplexF64, 3}:
[:, :, 1] =
 0.282095+0.0im       0.0+0.0im  0.0+0.0im
 0.488576+0.0im       0.0+0.0im  0.0+0.0im
 0.630679+0.0im       0.0+0.0im  0.0+0.0im
 0.746107+0.0im       0.0+0.0im  0.0+0.0im
  0.84582+0.0im       0.0+0.0im  0.0+0.0im
 0.934832+0.0im       0.0+0.0im  0.0+0.0im
  1.01593+0.0im       0.0+0.0im  0.0+0.0im
  1.09087+0.0im       0.0+0.0im  0.0+0.0im
  1.16081+0.0im       0.0+0.0im  0.0+0.0im
  1.22659+0.0im       0.0+0.0im  0.0+0.0im
  1.28882+0.0im       0.0+0.0im  0.0+0.0im
  1.34798+0.0im       0.0+0.0im  0.0+0.0im
  1.40444+0.0im       0.0+0.0im  0.0+0.0im
         ⋮                       
      0.0+0.0im  -1.50692+0.0im  0.0-1.50692im
      0.0+0.0im  -1.50875+0.0im  0.0-1.50875im
      0.0+0.0im  -1.51039+0.0im  0.0-1.51039im
      0.0+0.0im  -1.51186+0.0im  0.0-1.51186im
      0.0+0.0im  -1.51314+0.0im  0.0-1.51314im
      0.0+0.0im  -1.51424+0.0im  0.0-1.51424im
      0.0+0.0im  -1.51517+0.0im  0.0-1.51517im
      0.0+0.0im  -1.51592+0.0im  0.0

In [61]:
φ.+dφ

10-element Vector{Float64}:
 1.184885942919513
 1.1866962103042489
 1.1784405008562786
 1.1782985326275113
 1.1790953864088611
 1.1839966137867675
 1.1824868418720687
 1.187186938843399
 1.1793807964482965
 1.1837689100106155

In [71]:
alphas = zeros(size(dθ))
betas = zeros(size(dθ))
gammas = zeros(size(dθ))

for i in eachindex(dθ)
    err, (alphas[i], betas[i], gammas[i]) = check_split(φ, θ, dφ[i], dθ[i], ψ[i])
    if err >= 1e0
        @show err
    end
end

In [73]:
@show α_local == alphas
@show β_local == betas
@show γ_local == gammas

α_local == alphas = true
β_local == betas = true
γ_local == gammas = true


true

In [50]:
@time A = local_effective_wignerD(cb, cc, alphas, betas, gammas)
@time A_conj = local_effective_wignerD_conj(cb, cc, alphas, betas, gammas)
@time A2_conj = local_effective_wignerD_conj_reduced(cb, cc, alphas, betas, gammas; τ=5)
;

  0.695295 seconds (27 allocations: 3.442 MiB)
  0.788696 seconds (27 allocations: 3.442 MiB)
  0.081685 seconds (23 allocations: 493.266 KiB)


In [51]:
@show maximum(abs.(A_conj .- conj.(A)))
@show maximum(abs.(A2_conj .- conj.(A)))
@show maximum(abs.(A_conj .- (A2_conj) ))

maximum(abs.(A_conj .- conj.(A))) = 2.374296708786948e-14
maximum(abs.(A2_conj .- conj.(A))) = 3.935821241779829e-14
maximum(abs.(A_conj .- A2_conj)) = 3.935821241779829e-14


3.935821241779829e-14

In [10]:
hcat(blm_harmonic.blmT.alm,blm_harmonic.blmE.alm,blm_harmonic.blmB.alm)

285×3 Matrix{ComplexF64}:
 0.282095+0.0im       0.0+0.0im  0.0+0.0im
 0.488576+0.0im       0.0+0.0im  0.0+0.0im
 0.630679+0.0im       0.0+0.0im  0.0+0.0im
 0.746107+0.0im       0.0+0.0im  0.0+0.0im
  0.84582+0.0im       0.0+0.0im  0.0+0.0im
 0.934832+0.0im       0.0+0.0im  0.0+0.0im
  1.01593+0.0im       0.0+0.0im  0.0+0.0im
  1.09087+0.0im       0.0+0.0im  0.0+0.0im
  1.16081+0.0im       0.0+0.0im  0.0+0.0im
  1.22659+0.0im       0.0+0.0im  0.0+0.0im
  1.28882+0.0im       0.0+0.0im  0.0+0.0im
  1.34798+0.0im       0.0+0.0im  0.0+0.0im
  1.40444+0.0im       0.0+0.0im  0.0+0.0im
         ⋮                       
      0.0+0.0im  -1.50692+0.0im  0.0-1.50692im
      0.0+0.0im  -1.50875+0.0im  0.0-1.50875im
      0.0+0.0im  -1.51039+0.0im  0.0-1.51039im
      0.0+0.0im  -1.51186+0.0im  0.0-1.51186im
      0.0+0.0im  -1.51314+0.0im  0.0-1.51314im
      0.0+0.0im  -1.51424+0.0im  0.0-1.51424im
      0.0+0.0im  -1.51517+0.0im  0.0-1.51517im
      0.0+0.0im  -1.51592+0.0im  0.0-1.51592im
     

3×1149×1 Array{ComplexF64, 3}:
[:, :, 1] =
 1.0+1.0im  1.0+1.0im  1.0+1.0im  …  1.0+1.0im  1.0+1.0im  1.0+1.0im
 1.0+1.0im  1.0+1.0im  1.0+1.0im     1.0+1.0im  1.0+1.0im  1.0+1.0im
 1.0+1.0im  1.0+1.0im  1.0+1.0im     1.0+1.0im  1.0+1.0im  1.0+1.0im

# Convolution test

In [ ]:
@inline function convolver_(alm, blm, wignerD)
    
    for l in 0:cp.lmax
        result_1 += transpose(alm_new[1,l+1,lmax+1-l:lmax+1+l])*conj(eff_wignerD[l+1][:,:]*blm_new[1,l+1,lmax+1-l:lmax+1+l])
        result_2 += transpose(alm_new[2,l+1,lmax+1-l:lmax+1+l])*conj(eff_wignerD[l+1][:,:]*blm_new[2,l+1,lmax+1-l:lmax+1+l])
        result_3 += transpose(alm_new[3,l+1,lmax+1-l:lmax+1+l])*conj(eff_wignerD[l+1][:,:]*blm_new[3,l+1,lmax+1-l:lmax+1+l])
    end

end



convolver_ (generic function with 1 method)

In [15]:
cb.blm

1×3×1149 Array{ComplexF64, 3}:
[:, :, 1] =
 1.0+1.0im  1.0+1.0im  1.0+1.0im

[:, :, 2] =
 1.0+1.0im  1.0+1.0im  1.0+1.0im

[:, :, 3] =
 1.0+1.0im  1.0+1.0im  1.0+1.0im

;;; … 

[:, :, 1147] =
 1.0+1.0im  1.0+1.0im  1.0+1.0im

[:, :, 1148] =
 1.0+1.0im  1.0+1.0im  1.0+1.0im

[:, :, 1149] =
 1.0+1.0im  1.0+1.0im  1.0+1.0im

In [8]:
cb.lmax

383

In [13]:
alm_idx(lmax = cb.lmax,mmax =2, l=cb.lmax, m=2)

1149

In [5]:
cb.blm

1×3×1149 Array{ComplexF64, 3}:
[:, :, 1] =
 1.0+1.0im  1.0+1.0im  1.0+1.0im

[:, :, 2] =
 1.0+1.0im  1.0+1.0im  1.0+1.0im

[:, :, 3] =
 1.0+1.0im  1.0+1.0im  1.0+1.0im

;;; … 

[:, :, 1147] =
 1.0+1.0im  1.0+1.0im  1.0+1.0im

[:, :, 1148] =
 1.0+1.0im  1.0+1.0im  1.0+1.0im

[:, :, 1149] =
 1.0+1.0im  1.0+1.0im  1.0+1.0im

In [12]:
cs.alm

2×2 Matrix{ComplexF64}:
 1.0+1.0im  1.0+1.0im
 1.0+1.0im  1.0+1.0im